In [1]:
import pandas as pd
import numpy as np
import wandb
import os
import yaml
import torch

from transformers import AutoTokenizer, BartForConditionalGeneration
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback
from torch.utils.data import Dataset
from rouge import Rouge
from tqdm import tqdm


/root/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config File Load

In [2]:
# Config 파일 불러오기
config_path = './config.yaml'
with open(config_path, "r", encoding='utf-8') as file:
    config = yaml.safe_load(file)

print("Config Load 완료 세팅된 주요 값 확인")
print(f"- Enocder Max Lenth (대화문 95% 커버) : {config['tokenizer']['encoder_max_len']}")
print(f"- Decoder Max Lenth (요약문 커버) : {config['tokenizer']['decoder_max_len']}")
print(f"- 발굴된 특수 토큰 개수 : {len(config['tokenizer']['special_tokens'])}개")

Config Load 완료 세팅된 주요 값 확인
- Enocder Max Lenth (대화문 95% 커버) : 1024
- Decoder Max Lenth (요약문 커버) : 50
- 발굴된 특수 토큰 개수 : 22개


## Model Load & 임베딩 Layer 확장

📒 전략 <br>
- 우리가 허깅페이스에서 이미 학습된 Ko-BART 모델을 불러온건데, 거기에는 우리 데이터에서 정제한 데이터 내용이 없음
- 그래서 모델의 임베딩 레이어 크기를 새 단어장 크기만큼 물리적으로 늘려야함 -> 결국엔 사전에 우리 데이터를 추가하는 로직임

In [3]:
model_name = config['general']['model_name']
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"💻 Loding Device : {device}")

💻 Loding Device : cuda:0


In [4]:
# 1. 모델이랑 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

print(f"특수 토큰 추가 전 Tokenizer 단어장 크기 : {len(tokenizer)}")

# 2. 우리가 찾았던 22개 특수 토큰을 Tokenizer 사전에 등록
special_tokens_dict = {'additional_special_tokens': config['tokenizer']['special_tokens']}
tokenizer.add_special_tokens(special_tokens_dict)

# 3. 늘어난 단어장 크기에 맞춰서 모델의 임베딩 레이어 (뇌 용량) 확장
model.resize_token_embeddings(len(tokenizer))

# 모델을 GPU 로 이동
model.to(device)

print(f"특수 토큰 추가 후 Tokenizer 단어장 크기 : {len(tokenizer)}")
print("Tokenizer 와 Model Setting 그리고 임베딩 확장 완료")

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 262/262 [00:00<00:00, 20497.04it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion

특수 토큰 추가 전 Tokenizer 단어장 크기 : 30000
특수 토큰 추가 후 Tokenizer 단어장 크기 : 30022
Tokenizer 와 Model Setting 그리고 임베딩 확장 완료


In [5]:
# 모델이 새롭게 들어와야 새로운 적용값들이 적용되기에 해당 사항 Debug Code
test_text = "[화자1]은 [화자2]에게 감기에 안 걸렸다고 말했다."
tokens = tokenizer.tokenize(test_text)
print(f"토크나이징 테스트 결과 : {tokens}")

토크나이징 테스트 결과 : ['[화자1]', '▁은', '▁', '[화자2]', '▁', '에게', '▁감', '기에', '▁안', '▁걸', '렸다고', '▁말했다.']


## Text -> Number

In [6]:
# 1. 전처리 Class (BART 입력 형태에 맞게 텍스트에 <s>, </s> 토큰을 붙여주는 역할)
class Preprocess:
    def __init__(self, bos_token: str, eos_token: str):
        self.bos_token = bos_token
        self.eos_token = eos_token

    def make_input(self, df):
        # 인코더 입력 : 대화문 원본 (이미 [주제: ...]가 붙어있음)
        encoder_input = df['dialogue'].tolist()

        # 디코더 입력 : <s> + 요약문 (학습 시 모델에게 "여기서부터 요약 시작이다" 라고 알려줌)
        decoder_input = df['summary'].apply(lambda x : self.bos_token + str(x)).tolist()

        # 정답 레이블 : 요약문 + </s> (학습할 때 모델한테 "여기서 문장이 끝남" 이라고 알려줌)
        decoder_output = df['summary'].apply(lambda x : str(x) + self.eos_token).tolist()

        return encoder_input, decoder_input, decoder_output

In [7]:
# 2. PyTorch DataSet 클래스 (모델 <- 데이터 를 하나씩 넣어주기)
class CustomDataset(Dataset):
    def __init__(self, encoder_input, decoder_input, labels, length):
        self.encoder_input = encoder_input
        self.decoder_input = decoder_input
        self.labels = labels
        self.length = length
    
    def __getitem__(self, idx):
        # 텐서 복사, 분리 (안전한 학습을 위해 detach 사용)
        item = {key: torch.tensor(val[idx]) for key, val in self.encoder_input.items()}
        item2 = {key: torch.tensor(val[idx]) for key, val in self.decoder_input.items()}

        # BART 모델의 입력 규격에 맞게 이름 변경
        item['decoder_input_ids'] = item2['input_ids']
        item['decoder_attention_mask'] = item2['attention_mask']
        item['labels'] = torch.tensor(self.labels['input_ids'][idx])

        return item
    
    def __len__(self):
        return self.length

In [8]:
# ==========================================
# 📊 ROUGE 평가 함수 및 W&B 세팅 (유지)
# ==========================================
def compute_metrics(pred):
    rouge = Rouge()
    predictions = pred.predictions
    labels = pred.label_ids
    predictions[predictions == -100] = tokenizer.pad_token_type_id
    labels[labels == -100] = tokenizer.pad_token_id
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=False, clean_up_tokenization_spaces=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=False, clean_up_tokenization_spaces=True)
    remove_tokens = config['inference']['remove_tokens']
    for token in remove_tokens:
        decoded_preds = [sentence.replace(token, " ") for sentence in decoded_preds]
        decoded_labels = [sentence.replace(token, " ") for sentence in decoded_labels]
    results = rouge.get_scores(decoded_preds, decoded_labels, avg=True)
    result = {key: value["f"] for key, value in results.items()}
    result["final_result"] = (result["rouge-1"] + result["rouge-2"] + result["rouge-l"]) / 3.0 * 100.0
    return result

# 💡 W&B 이름 뒤에 "_2stage"를 붙여서 기존 실험과 분리!
wandb.init(entity=config['wandb']['entity'], project=config['wandb']['project'], name=config['wandb']['name'] + "_2stage")
os.environ["WANDB_LOG_MODEL"] = "true"
os.environ["WANDB_WATCH"] = "false"

# ==========================================
# 🚀 [STAGE 1] 거대 증강 데이터 학습 (Pre-training)
# ==========================================
import gc
torch.cuda.empty_cache()
gc.collect()

# 🚨 [추가] 여기서 무조건 백지 모델로 강제 초기화! (주피터 꼬임 완벽 방지)
print("🧠 모델 뇌를 순정 상태로 강제 초기화합니다...")
model_name = config['general']['model_name'] # config.yaml에 있는 digit82/kobart-summarization
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

special_tokens_dict = {'additional_special_tokens': config['tokenizer']['special_tokens']}
tokenizer.add_special_tokens(special_tokens_dict)
model.resize_token_embeddings(len(tokenizer))
model.to(device)
print("✅ 뇌 세척 완료! 완전히 순수해진 모델로 Stage 1을 시작합니다.")


data_path = config['general']['data_path']
preprocessor = Preprocess(config['tokenizer']['bos_token'], config['tokenizer']['eos_token'])

# 💡 세션 B가 준 2.4만 건 데이터 로드
train_df_stage1 = pd.read_csv(os.path.join(data_path, 'train_aug_24k_ready.csv'))
val_df = pd.read_csv(os.path.join(data_path, 'dev_processed.csv')) # 검증은 순정 Dev 유지

enc_in_1, dec_in_1, dec_out_1 = preprocessor.make_input(train_df_stage1)
enc_in_v, dec_in_v, dec_out_v = preprocessor.make_input(val_df)

tok_enc_1 = tokenizer(enc_in_1, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['encoder_max_len'])
tok_dec_in_1 = tokenizer(dec_in_1, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['decoder_max_len'])
tok_dec_out_1 = tokenizer(dec_out_1, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['decoder_max_len'])

tok_enc_v = tokenizer(enc_in_v, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['encoder_max_len'])
tok_dec_in_v = tokenizer(dec_in_v, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['decoder_max_len'])
tok_dec_out_v = tokenizer(dec_out_v, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['decoder_max_len'])

train_dataset_stage1 = CustomDataset(tok_enc_1, tok_dec_in_1, tok_dec_out_1, len(enc_in_1))
val_dataset = CustomDataset(tok_enc_v, tok_dec_in_v, tok_dec_out_v, len(enc_in_v))

training_args_stage1 = Seq2SeqTrainingArguments(
    output_dir=os.path.join(config['general']['output_dir'], "model_save/stage1_checkpoints"),
    num_train_epochs=5,               # 💡 과적합 방지를 위해 7 -> 5로 다이어트!
    learning_rate=1.5e-05,
    per_device_train_batch_size=config['training']['per_device_train_batch_size'],
    per_device_eval_batch_size=config['training']['per_device_eval_batch_size'],
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=50
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=3)

trainer_stage1 = Seq2SeqTrainer(
    model=model,
    args=training_args_stage1,
    train_dataset=train_dataset_stage1,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping]
)

print("🔥 [Stage 1] 2.4만 건 학습 시작!")
trainer_stage1.train()

stage1_model_path = config['general']['output_dir'] + "model_save/stage1_model"
trainer_stage1.save_model(stage1_model_path)
print(f"✅ [Stage 1] 완료! 모델이 {stage1_model_path} 에 저장되었습니다.")

# ==========================================
# 🎯 [STAGE 2] 순정 데이터 영점 조절 (Fine-tuning)
# ==========================================
torch.cuda.empty_cache()
gc.collect()

# 💡 세션 B가 준 1.2만 건 순정 족보 로드
train_df_stage2 = pd.read_csv(os.path.join(data_path, 'train_pure_12k_ready.csv'))

enc_in_2, dec_in_2, dec_out_2 = preprocessor.make_input(train_df_stage2)

tok_enc_2 = tokenizer(enc_in_2, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['encoder_max_len'])
tok_dec_in_2 = tokenizer(dec_in_2, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['decoder_max_len'])
tok_dec_out_2 = tokenizer(dec_out_2, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['decoder_max_len'])

train_dataset_stage2 = CustomDataset(tok_enc_2, tok_dec_in_2, tok_dec_out_2, len(enc_in_2))

training_args_stage2 = Seq2SeqTrainingArguments(
    output_dir=os.path.join(config['general']['output_dir'], "model_save/stage2_checkpoints"),
    num_train_epochs=2,
    learning_rate=3.0e-06, # 💡 극소 LR로 족보만 훑기!
    per_device_train_batch_size=config['training']['per_device_train_batch_size'],
    per_device_eval_batch_size=config['training']['per_device_eval_batch_size'],
    warmup_ratio=0.0,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=50
)

early_stopping_stage2 = EarlyStoppingCallback(early_stopping_patience=2)

trainer_stage2 = Seq2SeqTrainer(
    model=model, # 💡 Stage 1에서 똑똑해진 그 뇌를 그대로 이어서 학습!
    args=training_args_stage2,
    train_dataset=train_dataset_stage2,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping_stage2]
)

print("🔥 [Stage 2] 1.2만 건 벼락치기 영점 조절 시작!")
trainer_stage2.train()

final_model_path = config['general']['output_dir'] + "model_save/best_model_ultimate"
trainer_stage2.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
wandb.finish()
print(f"✅ [Stage 2] 완료! 궁극의 최종 모델이 {final_model_path} 에 저장되었습니다.")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /data/ephemeral/home/.netrc.


wandb: Currently logged in as: gam10678 (gam10678-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


🧠 모델 뇌를 순정 상태로 강제 초기화합니다...


You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 262/262 [00:00<00:00, 22582.92it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ 뇌 세척 완료! 완전히 순수해진 모델로 Stage 1을 시작합니다.


...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[RANK 0] Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/tmp/ipykernel_715443/2913228691.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encoder_input.items()}
/tmp/ipykernel_715443/2913228691.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item2 = {key: torch.tensor(val[idx]) for key, val in self.decoder_input.items()}
/tmp/ipykernel_715443/2913228691.py:17: UserWarning: To copy construct from a te

🔥 [Stage 2] 1.2만 건 벼락치기 영점 조절 시작!


Epoch,Training Loss,Validation Loss,Rouge-1,Rouge-2,Rouge-l,Final Result
1,0.810517,1.138006,0.327013,0.113937,0.307656,24.953535
2,0.780248,1.141494,0.326483,0.115718,0.307290,24.983056


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.84it/s]
/tmp/ipykernel_715443/2913228691.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encoder_input.items()}
/tmp/ipykernel_715443/2913228691.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item2 = {key: torch.tensor(val[idx]) for key, val in self.decoder_input.items()}
/tmp/ipykernel_715443/2913228691.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels['input_ids'][id

✅ [Stage 2] 완료! 궁극의 최종 모델이 ./model_save/best_model_ultimate 에 저장되었습니다.


## Inference Check

In [ ]:
# 1. 최고 성능 모델 & 토크나이저 불러오기
best_model_path = config['general']['output_dir'] + "model_save/best_model_ultimate"
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(f"{best_model_path} 에서 불러옵니다.")
infer_tokenizer = AutoTokenizer.from_pretrained(best_model_path)
infer_model = BartForConditionalGeneration.from_pretrained(best_model_path).to(device)

# 2. 검증 데이터 준비 (dev.csv)
data_path = config['general']['data_path']
val_df = pd.read_csv(os.path.join(data_path, 'dev_processed.csv'))

# 눈으로 확인 -> 앞의 3개 데이터만 뽑아서 Test
sample_df = val_df.head(3)
print("모델 추론 결과 확인해보기")
print("=" * 50)

# 3. 텍스트 생성
for idx, row in sample_df.iterrows():
    dialogue = row['dialogue']
    gold_summary = row['summary']

    # 3-1. 모델 입력 전처리 (프롬프트 부착 + 화자 치환)
    topic = row.get('topic', '일상 대화')
    prompted_dialogue = f"[주제: {topic}]\n" + dialogue

    # for i in range(1, 8):
        # prompted_dialogue = prompted_dialogue.replace(f"#Person{i}#", f"[화자{i}]")

    # 규칙대로 프롬프트 결합 (대화문만)
    input_text = prompted_dialogue

    # 모델 입력 형태로 토크나이징
    inputs = infer_tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)

    # Beam Search 등 모델 생성 옵션 적용
    summary_ids = infer_model.generate(
        inputs['input_ids'],
        num_beams=8,        # 수정사항 BeanSearch 값을 config[][].. 로 주는게 아닌 직접 숫자형을 매핑해서 깐깐하게
        max_length=50,
        early_stopping=True,
        no_repeat_ngram_size=2,
        # 수정사항 길이 패널티 추가 
        length_penalty=1.2,
        repetition_penalty=1.2
    )

    # 예측된 숫자를 다시 텍스트로 변환
    pred_summary = infer_tokenizer.decode(summary_ids[0], skip_special_tokens=False)

    # 클렌징
    remove_tokens = ['<usr>', '<s>', '</s>', '<pad>']
    for token in remove_tokens:
        pred_summary = pred_summary.replace(token, "").strip()

    # 출력후 전처리 (롤백)
    for i in range(1, 8):
        # 1. 정상적으로 [화자1] 형태일 때 완벽 치환
        pred_summary = pred_summary.replace(f"[화자{i}]", f"#Person{i}#")
        # 2. 혹시 모델이 괄호를 떼먹고 '화자1'만 뱉었을 경우 대비
        pred_summary = pred_summary.replace(f"화자{i}", f"#Person{i}#")
        # 3. 모델이 대괄호를 이중으로 쳤을 경우 대비 ([[화자1]] -> [#Person1#])
        pred_summary = pred_summary.replace(f"[#Person{i}#]", f"#Person{i}#")

    print(f"[Sample {idx + 1}]")
    print(f"원본 대화 (앞부분 일부) : {dialogue[:150]}...")
    print(f"정답 요약 : {gold_summary}")
    print(f"예측 요약 : {pred_summary}")

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


./model_save/best_model_ultimate 에서 불러옵니다.


Loading weights: 100%|██████████| 260/260 [00:00<00:00, 6895.95it/s]


모델 추론 결과 확인해보기
[Sample 1]
원본 대화 (앞부분 일부) : [화자1]: 안녕하세요, 오늘 기분이 어떠세요?
[화자2]: 요즘 숨쉬기가 힘들어요.
[화자1]: 최근에 감기에 걸렸나요?
[화자2]: 아니요, 감기는 안 걸렸어요. 숨쉴 때 가슴이 답답해요.
[화자1]: 혹시 알고 있는 알레르기 있으세요?
[화자2]: 아니요, 특별히...
정답 요약 : [화자2]는 숨쉬기 어려워합니다. 의사는 [화자2]에게 증상을 확인하고, 천식 검사를 위해 폐 전문의에게 가볼 것을 권합니다.
예측 요약 : #Person2# 는 최근 숨쉬기 어려움을 겪고 있으며, 천식 검사를 위해 폐 전문의에게 가볼 것을 제안합니다.
[Sample 2]
원본 대화 (앞부분 일부) : [화자1]: 야 Jimmy, 오늘 좀 이따 운동하러 가자.
[화자2]: 그래, 몇 시에 갈래?
[화자1]: 3시 30분 어때?
[화자2]: 좋아. 오늘은 다리랑 팔 운동하는 날이야.
[화자1]: 나 아까 농구해서 다리가 좀 아파. 오늘은 팔이랑 복근 운동하자.
[화자2...
정답 요약 : [화자1]는 Jimmy를 운동하러 초대하고 팔과 복근 운동을 하도록 설득합니다.
예측 요약 : #Person1# 은 야 Jimmy에게 운동 계획을 묻고, 야는 오후 3시 30분에 체육관에서 운동하자고 제안한다.
[Sample 3]
원본 대화 (앞부분 일부) : [화자1]: 나 건강에 안 좋은 음식 좀 그만 먹어야겠어. 
[화자2]: 맞아, 무슨 말인지 알아. 나도 요즘 건강하게 먹으려고 하거든. 
[화자1]: 요즘은 뭐 먹어? 
[화자2]: 주로 과일이랑 채소, 닭고기 먹지. 
[화자1]: 그게 다야? 
[화자2]: 거의 그...
정답 요약 : [화자1]은 건강에 안 좋은 음식을 그만 먹기로 결심하고, [화자2]는 자신의 건강한 식단을 [화자1]에게 공유합니다.
예측 요약 : #Person1# 과 #Person2# 는 건강한 식단에 대해 논의합니다. #Person1# 은 주로 과일과 채소, 닭고기를 먹으며, 구워 먹으

: 

## Submission

In [10]:
# 1. 테스트 데이터 불러오기 (대회 제공 test 파일)
test_df = pd.read_csv(os.path.join(config['general']['data_path'], 'test_processed.csv'))

# 2. 결과 저장을 위한 리스트
predicted_summaries = []

infer_model.eval()  # 평가모드 전환

# 3. 한 줄씩 추론 진행
with torch.no_grad():
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        # 훈련할 때 똑같은 하드 프롬프트 부착
        topic = row.get('topic', '일상 대화')
        prompted_dialogue = f"[주제: {topic}]\n" + row['dialogue']

        # 💡 피드백 반영: for문을 이용해 1~7번 화자까지 다이내믹하게 치환!
        # for i in range(1, 8):
            # prompted_dialogue = prompted_dialogue.replace(f"#Person{i}#", f"[화자{i}]")

        # 토크나이징 및 텐서 변환
        inputs = infer_tokenizer(prompted_dialogue, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)

        # 찾아본 최적의 디코딩 파라미터 적용
        summary_ids = infer_model.generate(
            inputs['input_ids'],
            num_beams=8,
            length_penalty=1.2,
            repetition_penalty=1.2,
            max_length=50,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

        # 디코딩 (skip_special_tokens=False 유지)
        pred_summary = infer_tokenizer.decode(summary_ids[0], skip_special_tokens=False)

        # 클렌징 (메모리 버그 방지로 인해 Config 에서 불러오는 것 보다 강제 하드코딩)
        remove_tokens_safe = ['<usr>', '<s>', '</s>', '<pad>']
        for token in remove_tokens_safe:
            pred_summary = pred_summary.replace(token, "").strip()
        
        # 채점기를 위해 [화자1] -> #Person1# 복구
        for i in range(1, 8):
            pred_summary = pred_summary.replace(f"[화자{i}]", f"#Person{i}#")
            pred_summary = pred_summary.replace(f"화자{i}", f"#Person{i}#")
            pred_summary = pred_summary.replace(f"[#Person{i}#]", f"#Person{i}#")

        predicted_summaries.append(pred_summary.strip())

# 4. 제출용 DataFrame 만들기 및 CSV Save
submission_df = pd.DataFrame({
    'fname': test_df['fname'],
    'summary': predicted_summaries
})

submission_path = '../data/03_submission.csv'
submission_df.to_csv(submission_path, index=False, encoding='utf-8-sig')

print(f"제출 파일 생성 완료 : {submission_path}")

  0%|          | 0/499 [00:00<?, ?it/s]

100%|██████████| 499/499 [01:39<00:00,  5.01it/s]

제출 파일 생성 완료 : ../data/03_submission.csv


In [11]:
import pandas as pd
import os

# 1. 데이터 로드 (경로는 본인 환경에 맞게 살짝 수정해 줘!)
data_path = '../data/train_aug_24k_ready.csv' # 또는 config['general']['data_path'] 활용
df = pd.read_csv(data_path)

print("="*50)
print("📊 [Step 1] 전체 데이터 볼륨 및 결측치 체크")
print("="*50)
print(f"✅ 전체 데이터 개수: {len(df)}개")

# 🚨 네가 가장 의심했던 부분! Summary가 아예 비어있는(NaN) 데이터가 있는지 확인
null_summary_count = df['summary'].isnull().sum()
print(f"🚨 요약문(summary)이 비어있는 데이터 개수: {null_summary_count}개")

# Summary가 'None'이나 빈 문자열 등 이상한 값으로 채워진 건 없는지 확인
empty_string_count = len(df[df['summary'].str.strip() == ''])
print(f"🚨 요약문이 빈 칸('')으로 되어있는 데이터 개수: {empty_string_count}개")


print("\n" + "="*50)
print("🕵️‍♂️ [Step 2] 증강 데이터(하위 1.2만 건 추정) 랜덤 5개 육안 검수")
print("="*50)

# 하위 12,000개(증강 데이터) 중에서 랜덤하게 5개를 뽑아서 눈으로 직접 읽어보기
augmented_sample = df.tail(12000).sample(5, random_state=42)

for idx, row in augmented_sample.iterrows():
    print(f"\n[데이터 Index: {idx}]")
    
    # 대화문은 너무 길면 보기 힘드니 200자까지만 자르기
    dialogue = str(row['dialogue'])
    dialogue_preview = dialogue[:200] + "..." if len(dialogue) > 200 else dialogue
    
    print(f"🗣️ 대화문 (미리보기): {dialogue_preview}")
    print(f"📝 요약문 (정답지): {row['summary']}")
    print("-" * 50)

📊 [Step 1] 전체 데이터 볼륨 및 결측치 체크
✅ 전체 데이터 개수: 24271개
🚨 요약문(summary)이 비어있는 데이터 개수: 0개
🚨 요약문이 빈 칸('')으로 되어있는 데이터 개수: 0개

🕵️‍♂️ [Step 2] 증강 데이터(하위 1.2만 건 추정) 랜덤 5개 육안 검수

[데이터 Index: 14206]
🗣️ 대화문 (미리보기): #Person1#: 예약하셨나요, 손님?  
#Person2#: 아니요, 아직 안 했습니다.  
#Person1#: 죄송합니다만, 현재 만석입니다. 약 30분 정도 기다리셔야 합니다. 기다리는 동안 라운지에서 음료를 드시겠어요?  
#Person2#: 괜찮습니다. 나중에 다시 오겠습니다. 두 명 테이블 예약할 수 있을까요?  
#Person1#: 네, 가능...
📝 요약문 (정답지): Bruce는 식당이 만석이라 약 30분 후로 두 명의 자리를 예약하고 나중에 다시 오기로 합니다.
--------------------------------------------------

[데이터 Index: 18765]
🗣️ 대화문 (미리보기): #Person1#: 안녕, 잭. 누구에게 편지를 쓰고 있는 거야?  
#Person2#: 부모님께 이번 여름에 여기 있을 거라고 말씀드리는 중이에요.  
#Person1#: 집에 안 가는 거야? 너랑 가족이 일본으로 여행 간다고 들었어.  
#Person2#: 원래는 같이 가려고 했는데 다시 생각해보고 마음을 바꿨어요.  
#Person1#: 사랑하는 가족...
📝 요약문 (정답지): Jack은 방학 동안 일하는 경험을 얻기 위해 자원봉사를 하며 머물고, [화자1]은 집으로 떠나 하와이로 여행을 떠난다.
--------------------------------------------------

[데이터 Index: 13991]
🗣️ 대화문 (미리보기): #Person1#: 엄마, 오늘 저녁으로 뭐 만들 거예요?  
#Person2#: 카레라이스. 어때요?  
#Person1#: 좋